In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors

In [3]:
df = pd.read_csv("../data/delaney.csv")

df.head()

,Compound ID,measured log(solubility:mol/L),ESOL predicted log(solubility:mol/L),SMILES
0,"1,1,1,2-Tetrachloroethane",-2.18,-2.794,ClCC(Cl)(Cl)Cl
1,"1,1,1-Trichloroethane",-2.00,-2.232,CC(Cl)(Cl)Cl
2,"1,1,2,2-Tetrachloroethane",-1.74,-2.549,ClC(Cl)C(Cl)Cl
3,"1,1,2-Trichloroethane",-1.48,-1.961,ClCC(Cl)Cl
4,"1,1,2-Trichlorotrifluoroethane",-3.04,-3.077,FC(F)(Cl)C(F)(Cl)Cl


In [4]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1144, 4)

Columns:
['Compound ID', 'measured log(solubility:mol/L)', 'ESOL predicted log(solubility:mol/L)', 'SMILES']


In [5]:
print(df.isnull().sum())

Compound ID                             0
measured log(solubility:mol/L)          0
ESOL predicted log(solubility:mol/L)    0
SMILES                                  0
dtype: int64


In [6]:
df.describe()

,measured log(solubility:mol/L),ESOL predicted log(solubility:mol/L)
count,1144.000000,1144.000000
mean,-3.057997,-2.994776
std,2.096502,1.686520
min,-11.600000,-9.702000
25%,-4.332250,-3.962250
50%,-2.870500,-2.889000
75%,-1.600000,-1.846750
max,1.580000,1.091000


In [7]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDuplicate SMILES:", df["SMILES"].duplicated().sum())

Missing values:
Compound ID                             0
measured log(solubility:mol/L)          0
ESOL predicted log(solubility:mol/L)    0
SMILES                                  0
dtype: int64

Duplicate rows: 0

Duplicate SMILES: 16


In [8]:
duplicates = df[df["SMILES"].duplicated(keep=False)].sort_values("SMILES")

duplicates[[
    "Compound ID",
    "measured log(solubility:mol/L)",
    "SMILES"
]]

,Compound ID,measured log(solubility:mol/L),SMILES
1046,RTI 7,-3.535,C2c1ccccc1N(CCF)C(=O)c3ccccc23
1039,RTI 20,-4.799,C2c1ccccc1N(CCF)C(=O)c3ccccc23
288,3-Methyl-2-pentanol,-0.720,CCC(C)CCO
289,3-Methyl-2-pentanol,-0.710,CCC(C)CCO
870,Nitramine,-3.561,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
607,Dinitramine,-5.470,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
668,Ethyl-p-aminobenzoate,-2.100,CCOC(=O)c1ccc(N)cc1
428,Benzocaine,-2.616,CCOC(=O)c1ccc(N)cc1
648,Ethofumesate,-3.420,CCOC2Oc1ccc(OS(C)(=O)=O)cc1C2(C)C
649,ethofumesate,-3.420,CCOC2Oc1ccc(OS(C)(=O)=O)cc1C2(C)C


In [9]:
duplicate_groups = (
    df.groupby("SMILES")["measured log(solubility:mol/L)"]
      .agg(["count", "min", "max", "mean", "std"])
      .query("count > 1")
      .sort_values("count", ascending=False)
)

duplicate_groups

,count,min,max,mean,std
SMILES,,,,,
C2c1ccccc1N(CCF)C(=O)c3ccccc23,2,-4.799,-3.535,-4.1670,0.893783
CCC(C)CCO,2,-0.720,-0.710,-0.7150,0.007071
CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O,2,-5.470,-3.561,-4.5155,1.349867
CCOC(=O)c1ccc(N)cc1,2,-2.616,-2.100,-2.3580,0.364867
CCOC2Oc1ccc(OS(C)(=O)=O)cc1C2(C)C,2,-3.420,-3.420,-3.4200,0.000000
CN(C)C(=O)Nc1ccc(C)c(Cl)c1,2,-3.483,-3.460,-3.4715,0.016263
CN(C)C(=O)Nc1cccc(c1)C(F)(F)F,2,-3.430,-3.320,-3.3750,0.077782
ClC(Cl)=C(c1ccc(Cl)cc1)c2ccc(Cl)cc2,2,-6.900,-6.900,-6.9000,0.000000
ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl,2,-6.290,-6.180,-6.2350,0.077782


In [10]:
print("Number of duplicated SMILES:", len(duplicate_groups))
print("Number of observations involved:", duplicate_groups["count"].sum())

Number of duplicated SMILES: 16
Number of observations involved: 32


In [11]:
print(
    "Conflicting duplicate structures:",
    (duplicate_groups["min"] != duplicate_groups["max"]).sum()
)

Conflicting duplicate structures: 12


In [12]:
conflicting = duplicate_groups[
    duplicate_groups["min"] != duplicate_groups["max"]
]

conflicting

,count,min,max,mean,std
SMILES,,,,,
C2c1ccccc1N(CCF)C(=O)c3ccccc23,2,-4.799,-3.535,-4.1670,0.893783
CCC(C)CCO,2,-0.720,-0.710,-0.7150,0.007071
CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O,2,-5.470,-3.561,-4.5155,1.349867
CCOC(=O)c1ccc(N)cc1,2,-2.616,-2.100,-2.3580,0.364867
CN(C)C(=O)Nc1ccc(C)c(Cl)c1,2,-3.483,-3.460,-3.4715,0.016263
CN(C)C(=O)Nc1cccc(c1)C(F)(F)F,2,-3.430,-3.320,-3.3750,0.077782
ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl,2,-6.290,-6.180,-6.2350,0.077782
Clc1ccc(cc1)c2cc(Cl)ccc2Cl,2,-6.270,-6.250,-6.2600,0.014142
Clc1ccc(cc1)c2cccc(Cl)c2Cl,2,-6.290,-6.260,-6.2750,0.021213


In [14]:
df[df["SMILES"].isin(conflicting.index)][
    [
        "Compound ID",
        "measured log(solubility:mol/L)",
        "SMILES"
    ]
].sort_values("SMILES")

,Compound ID,measured log(solubility:mol/L),SMILES
1039,RTI 20,-4.799,C2c1ccccc1N(CCF)C(=O)c3ccccc23
1046,RTI 7,-3.535,C2c1ccccc1N(CCF)C(=O)c3ccccc23
288,3-Methyl-2-pentanol,-0.720,CCC(C)CCO
289,3-Methyl-2-pentanol,-0.710,CCC(C)CCO
870,Nitramine,-3.561,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
607,Dinitramine,-5.470,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
428,Benzocaine,-2.616,CCOC(=O)c1ccc(N)cc1
668,Ethyl-p-aminobenzoate,-2.100,CCOC(=O)c1ccc(N)cc1
504,Chlorotoluron,-3.460,CN(C)C(=O)Nc1ccc(C)c(Cl)c1
509,Chlortoluron,-3.483,CN(C)C(=O)Nc1ccc(C)c(Cl)c1


In [13]:
from rdkit import Chem

molecules = df["SMILES"].apply(Chem.MolFromSmiles)

print("Invalid SMILES:", molecules.isna().sum())

Invalid SMILES: 0


In [ ]:
duplicate_audit = [
    {
        "records": "RTI 7 / RTI 20",
        "category": "D",
        "issue": "Same SMILES, substantially different measured values",
        "action": "Investigate"
    },
    {
        "records": "3-Methyl-2-pentanol / 3-Methyl-2-pentanol",
        "category": "B",
        "issue": "Same compound, slightly different measurements",
        "action": "Investigate measurement handling"
    },
    {
        "records": "Nitramine / Dinitramine",
        "category": "C",
        "issue": "Different compound names assigned same SMILES",
        "action": "Verify/correct SMILES"
    },
    {
        "records": "Benzocaine / Ethyl-p-aminobenzoate",
        "category": "A",
        "issue": "Different names for same compound",
        "action": "Treat as same molecule"
    },
    {
        "records": "Chlorotoluron / Chlortoluron",
        "category": "A",
        "issue": "Name variation",
        "action": "Treat as same molecule"
    },
    {
        "records": "Fluometuron / Fluorometuron",
        "category": "A",
        "issue": "Name variation",
        "action": "Treat as same molecule"
    },
    {
        "records": "Dieldrin / Endrin",
        "category": "C",
        "issue": "Different compounds assigned same SMILES",
        "action": "Verify/correct SMILES"
    },
    {
        "records": "2,4,5-PCB / 2,4',5-PCB",
        "category": "C",
        "issue": "Different positional isomers assigned same SMILES",
        "action": "Verify/correct SMILES"
    },
    {
        "records": "2',3,4-PCB / 2,3,4'-PCB",
        "category": "C",
        "issue": "Different positional isomers assigned same SMILES",
        "action": "Verify/correct SMILES"
    },
    {
        "records": "2,4-PCB / 2,4'-PCB",
        "category": "C",
        "issue": "Different positional isomers assigned same SMILES",
        "action": "Verify/correct SMILES"
    },
    {
        "records": "o-Hydroxybenzamide / Salicylamide",
        "category": "A",
        "issue": "Different names for same compound",
        "action": "Treat as same molecule"
    },
    {
        "records": "Lactose / Maltose",
        "category": "C",
        "issue": "Different compounds assigned same SMILES",
        "action": "Verify/correct SMILES"
    }
]

In [16]:
audit_df = pd.DataFrame(duplicate_audit)
audit_df

,records,category,issue,action
0,RTI 7 / RTI 20,D,"Same SMILES, substantially different measured ...",Investigate
1,3-Methyl-2-pentanol / 3-Methyl-2-pentanol,B,"Same compound, slightly different measurements",Investigate measurement handling
2,Nitramine / Dinitramine,C,Different compound names assigned same SMILES,Verify/correct SMILES
3,Benzocaine / Ethyl-p-aminobenzoate,A,Different names for same compound,Treat as same molecule
4,Chlorotoluron / Chlortoluron,A,Name variation,Treat as same molecule
5,Fluometuron / Fluorometuron,A,Name variation,Treat as same molecule
6,Dieldrin / Endrin,C,Different compounds assigned same SMILES,Verify/correct SMILES
7,"2,4,5-PCB / 2,4',5-PCB",C,Different positional isomers assigned same SMILES,Verify/correct SMILES
8,"2',3,4-PCB / 2,3,4'-PCB",C,Different positional isomers assigned same SMILES,Verify/correct SMILES
9,"2,4-PCB / 2,4'-PCB",C,Different positional isomers assigned same SMILES,Verify/correct SMILES


In [22]:
category_c_names = [
    "Nitramine",
    "Dinitramine",
    "Dieldrin",
    "Endrin",
    "2,4,5-PCB",
    "2,4',5-PCB",
    "2',3,4-PCB",
    "2,3,4'-PCB",
    "2,4-PCB",
    "2,4'-PCB",
    "Lactose",
    "Maltose"
]

category_c = df[df["Compound ID"].isin(category_c_names)][
    ["Compound ID", "measured log(solubility:mol/L)", "SMILES"]
].copy()

category_c

,Compound ID,measured log(solubility:mol/L),SMILES
151,"2',3,4-PCB",-6.290,Clc1ccc(cc1)c2cccc(Cl)c2Cl
152,"2,3,4'-PCB",-6.260,Clc1ccc(cc1)c2cccc(Cl)c2Cl
166,"2,4,5-PCB",-6.270,Clc1ccc(cc1)c2cc(Cl)ccc2Cl
167,"2,4',5-PCB",-6.250,Clc1ccc(cc1)c2cc(Cl)ccc2Cl
181,"2,4'-PCB",-5.280,Clc1ccc(cc1)c2ccccc2Cl
182,"2,4-PCB",-5.250,Clc1ccc(cc1)c2ccccc2Cl
582,Dieldrin,-6.290,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
607,Dinitramine,-5.470,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
628,Endrin,-6.180,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
771,Lactose,-0.244,OCC1OC(OC2C(O)C(O)C(O)OC2CO)C(O)C(O)C1O


In [27]:
initial_audit_df = audit_df.copy()

In [28]:
final_audit = pd.DataFrame(columns=[
    "Compound ID",
    "Compound Name",
    "Category",
    "Issue",
    "Chemical Relationship",
    "Evidence",
    "Action",
    "Final Decision",
    "Corrected SMILES"
])

In [29]:
category_c

,Compound ID,measured log(solubility:mol/L),SMILES
151,"2',3,4-PCB",-6.290,Clc1ccc(cc1)c2cccc(Cl)c2Cl
152,"2,3,4'-PCB",-6.260,Clc1ccc(cc1)c2cccc(Cl)c2Cl
166,"2,4,5-PCB",-6.270,Clc1ccc(cc1)c2cc(Cl)ccc2Cl
167,"2,4',5-PCB",-6.250,Clc1ccc(cc1)c2cc(Cl)ccc2Cl
181,"2,4'-PCB",-5.280,Clc1ccc(cc1)c2ccccc2Cl
182,"2,4-PCB",-5.250,Clc1ccc(cc1)c2ccccc2Cl
582,Dieldrin,-6.290,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
607,Dinitramine,-5.470,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
628,Endrin,-6.180,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
771,Lactose,-0.244,OCC1OC(OC2C(O)C(O)C(O)OC2CO)C(O)C(O)C1O


In [30]:
nitramine_case = category_c[
    category_c["Compound ID"].isin(["Nitramine", "Dinitramine"])
]

dieldrin_case = category_c[
    category_c["Compound ID"].isin(["Dieldrin", "Endrin"])
]

pcb_case = category_c[
    category_c["Compound ID"].str.contains("PCB", case=False, na=False)
]

sugar_case = category_c[
    category_c["Compound ID"].isin(["Lactose", "Maltose"])
]

print("Nitramine/Dinitramine:")
display(nitramine_case)

print("\nDieldrin/Endrin:")
display(dieldrin_case)

print("\nPCBs:")
display(pcb_case)

print("\nLactose/Maltose:")
display(sugar_case)

Nitramine/Dinitramine:


,Compound ID,measured log(solubility:mol/L),SMILES
607,Dinitramine,-5.470,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
870,Nitramine,-3.561,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O



Dieldrin/Endrin:


,Compound ID,measured log(solubility:mol/L),SMILES
582,Dieldrin,-6.29,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl
628,Endrin,-6.18,ClC4=C(Cl)C5(Cl)C3C1CC(C2OC12)C3C4(Cl)C5(Cl)Cl



PCBs:


,Compound ID,measured log(solubility:mol/L),SMILES
151,"2',3,4-PCB",-6.29,Clc1ccc(cc1)c2cccc(Cl)c2Cl
152,"2,3,4'-PCB",-6.26,Clc1ccc(cc1)c2cccc(Cl)c2Cl
166,"2,4,5-PCB",-6.27,Clc1ccc(cc1)c2cc(Cl)ccc2Cl
167,"2,4',5-PCB",-6.25,Clc1ccc(cc1)c2cc(Cl)ccc2Cl
181,"2,4'-PCB",-5.28,Clc1ccc(cc1)c2ccccc2Cl
182,"2,4-PCB",-5.25,Clc1ccc(cc1)c2ccccc2Cl



Lactose/Maltose:


,Compound ID,measured log(solubility:mol/L),SMILES
771,Lactose,-0.244,OCC1OC(OC2C(O)C(O)C(O)OC2CO)C(O)C(O)C1O
781,Maltose,0.358,OCC1OC(OC2C(O)C(O)C(O)OC2CO)C(O)C(O)C1O


In [31]:
nitramine_case

,Compound ID,measured log(solubility:mol/L),SMILES
607,Dinitramine,-5.470,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
870,Nitramine,-3.561,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O


In [32]:
final_audit.loc[len(final_audit)] = {
    "Compound ID": "607",
    "Compound Name": "Dinitramine",
    "Category": "C",
    "Issue": "Shared SMILES with a chemically different compound",
    "Chemical Relationship": "Different compound from Nitramine",
    "Evidence": "Verified against PubChem",
    "Action": "Keep",
    "Final Decision": "Retain original structure",
    "Corrected SMILES": "CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O"
}

final_audit.loc[len(final_audit)] = {
    "Compound ID": "870",
    "Compound Name": "Nitramine",
    "Category": "C",
    "Issue": "Incorrectly assigned Dinitramine SMILES",
    "Chemical Relationship": "Different compound from Dinitramine",
    "Evidence": "Verified against PubChem",
    "Action": "Correct SMILES",
    "Final Decision": "Retain record with corrected structure",
    "Corrected SMILES": "CN(C1=C(C=C(C=C1[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-])[N+](=O)[O-]"
}

In [33]:
final_audit

,Compound ID,Compound Name,Category,Issue,Chemical Relationship,Evidence,Action,Final Decision,Corrected SMILES
0,607,Dinitramine,C,Shared SMILES with a chemically different comp...,Different compound from Nitramine,Verified against PubChem,Keep,Retain original structure,CCN(CC)c1c(cc(c(N)c1N(=O)=O)C(F)(F)F)N(=O)=O
1,870,Nitramine,C,Incorrectly assigned Dinitramine SMILES,Different compound from Dinitramine,Verified against PubChem,Correct SMILES,Retain record with corrected structure,CN(C1=C(C=C(C=C1[N+](=O)[O-])[N+](=O)[O-])[N+]...
